In [86]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# --- Configuration for data generation ---
num_students = 200
terms = ['202510', '202520', '202610', '202620'] # Fall 2025, Spring 2026, Fall 2026, Spring 2027
program_codes = ['Data Analytics', 'Natural Sciences', 'Business', 'Engineering', 'Nursing', 'Arts & Humanities']
ethnicities = ['White', 'Hispanic/Latino', 'Black/African American', 'Asian', 'International', 'Two or More Races']
min_credits = 9
max_credits = 18

# New: Certificate program configurations
certificate_programs = ['Cybersecurity Cert', 'Web Development Cert', 'Data Science Micro-Cert', 'Healthcare Admin Cert']

# --- 1. Create Banner Enrollment Data ---
all_banner_data = []
student_ids = [f'S{i:03d}' for i in range(1, num_students + 1)]

# Assign a random entry term and potential graduation/certificate term to each student
student_lifecycles = {}
for sid in student_ids:
    entry_term_idx = np.random.randint(0, len(terms) - 2) # Ensure student has at least 2 terms to persist
    entry_term = terms[entry_term_idx]

    graduated_term = None
    if np.random.rand() < 0.25:
        grad_term_idx = np.random.randint(entry_term_idx + 1, min(entry_term_idx + 3, len(terms)))
        if grad_term_idx < len(terms):
            graduated_term = terms[grad_term_idx]

    # New: Assign certificate pathway to about 30% of students
    cert_program = None
    cert_completed_term = None
    if np.random.rand() < 0.3:
        cert_program = np.random.choice(certificate_programs)
        if np.random.rand() < 0.6: # 60% chance of completing the certificate if pursuing
            # Cert completion can be in the same term or next term after entry_term
            cert_completed_term_idx = np.random.randint(entry_term_idx, min(entry_term_idx + 2, len(terms)))
            if cert_completed_term_idx < len(terms):
                cert_completed_term = terms[cert_completed_term_idx]

    # New: Assign transfer status to about 20% of students
    is_transfer_student = np.random.rand() < 0.2

    student_lifecycles[sid] = {
        'entry_term_idx': entry_term_idx,
        'graduated_term': graduated_term,
        'certificate_program': cert_program,
        'cert_completed_term': cert_completed_term,
        'is_transfer': is_transfer_student
    }

for sid in student_ids:
    entry_term_idx = student_lifecycles[sid]['entry_term_idx']
    graduated_term = student_lifecycles[sid]['graduated_term']
    cert_program = student_lifecycles[sid]['certificate_program']
    cert_completed_term = student_lifecycles[sid]['cert_completed_term']
    is_transfer_student = student_lifecycles[sid]['is_transfer']

    for i in range(entry_term_idx, len(terms)):
        current_term = terms[i]

        if graduated_term and current_term >= graduated_term:
            continue

        retained = True
        if graduated_term and terms[i] == graduated_term:
            retained = False
        elif i < len(terms) - 1:
            if np.random.rand() < 0.2:
                retained = False
        else:
            retained = False

        program = np.random.choice(program_codes)
        ethnicity = np.random.choice(ethnicities)
        credits = np.random.randint(min_credits, max_credits + 1)

        # New: Certificate status for the current term
        current_cert_status = 'None'
        if cert_program:
            if cert_completed_term is not None and current_term == cert_completed_term:
                current_cert_status = 'Completed'
            elif cert_completed_term is None or current_term < cert_completed_term:
                current_cert_status = 'Pursuing'

        all_banner_data.append({
            'student_id': sid,
            'term_code': current_term,
            'program_code': program,
            'credits_attempted': credits,
            'ethnicity': ethnicity,
            'retained_next_term': retained,
            'graduated': (current_term == graduated_term) if graduated_term else False,
            'entry_term': terms[entry_term_idx],
            'certificate_program': cert_program,
            'certificate_status': current_cert_status,
            'is_transfer': is_transfer_student # Add new transfer status
        })

banner_df = pd.DataFrame(all_banner_data)
banner_df.to_csv('banner_enrollment.csv', index=False)

# --- 2. Create Navigate Risk Data ---
all_navigate_data = []
risk_levels = ['Low', 'Medium', 'High']
for _ in range(int(num_students * len(terms) * 0.3)):
    student = np.random.choice(student_ids)
    term = np.random.choice(terms)

    if not banner_df[(banner_df['student_id'] == student) & (banner_df['term_code'] == term)].empty:
        risk = np.random.choice(risk_levels, p=[0.6, 0.3, 0.1])
        date_offset = np.random.randint(1, 90)
        if term.endswith('10'):
            base_date = datetime(int(term[:4]), 9, 1)
        else:
            base_date = datetime(int(term[:4]), 1, 1)
        contact_date = (base_date + timedelta(days=date_offset)).strftime('%Y-%m-%d')

        all_navigate_data.append({
            'student_id': student,
            'term_code': term,
            'risk_level': risk,
            'last_contact_date': contact_date
        })

navigate_df = pd.DataFrame(all_navigate_data)
navigate_df.to_csv('navigate_risk.csv', index=False)

# --- 3. Create Ad Astra Scheduling Data (expanded) ---
ad_astra_course_ids = ['DATA101', 'DATA102', 'SCI110', 'SCI111', 'LEG101', 'BUS201', 'ENG300', 'NUR400']
expanded_ad_astra_data = []
for term_code in terms:
    for course_id in np.random.choice(ad_astra_course_ids, size=np.random.randint(4, 7), replace=False):
        section_id = f"SEC{np.random.randint(1,5):02d}"
        capacity = np.random.randint(20, 50)
        enrolled_count = np.random.randint(int(capacity * 0.5), int(capacity * 1.2))

        expanded_ad_astra_data.append({
            'term_code': term_code,
            'course_id': course_id,
            'section_id': section_id,
            'capacity': capacity,
            'enrolled_count': enrolled_count
        })

pd.DataFrame(expanded_ad_astra_data).to_csv('ad_astra_sections.csv', index=False)

# --- 4. Create Sectionizer Planning Data (new) ---
all_sectionizer_data = []
for term_code in terms:
    for course_id in np.random.choice(ad_astra_course_ids, size=np.random.randint(3, 6), replace=False):
        planned_sections = np.random.randint(1, 5)
        all_sectionizer_data.append({
            'term_code': term_code,
            'course_id': course_id,
            'planned_sections': planned_sections
        })

pd.DataFrame(all_sectionizer_data).to_csv('sectionizer_plans.csv', index=False)

print("CSVs generated successfully with richer data!")

CSVs generated successfully with richer data!


In [87]:
import sqlite3
import pandas as pd

# 1. Connect to a temporary in-memory database
conn = sqlite3.connect(':memory:')

# 2. Load your CSVs into SQL tables
pd.read_csv('banner_enrollment.csv').to_sql('banner_enrollment', conn, index=False)
pd.read_csv('navigate_risk.csv').to_sql('navigate_risk_signals', conn, index=False)
pd.read_csv('ad_astra_sections.csv').to_sql('ad_astra_sections', conn, index=False)
pd.read_csv('sectionizer_plans.csv').to_sql('sectionizer_plans', conn, index=False)

print("Database and tables initialized.")

Database and tables initialized.


In [89]:
# AUDIT: Duplicate Enrollment Check
# Modified to check for duplicate student_id + term_code pairs.
query_duplicate_check = """
SELECT
    student_id,
    term_code,
    COUNT(*) AS records_found
FROM banner_enrollment
GROUP BY student_id, term_code
HAVING COUNT(*) > 1;
"""
try:
    result_duplicate_check = pd.read_sql_query(query_duplicate_check, conn)
    print("Duplicate Enrollment Check Result:\n", result_duplicate_check)
except Exception as e:
    print(f"Error executing Duplicate Enrollment Check query: {e}")

Duplicate Enrollment Check Result:
 Empty DataFrame
Columns: [student_id, term_code, records_found]
Index: []


In [90]:
# METRIC: Program-wise Fall 2025 to Spring 2026 Persistence Rate
# Now uses the enhanced 'banner_enrollment' data to calculate persistence by program.
query_persistence_rate = """
SELECT
    e.program_code,
    COUNT(DISTINCT e.student_id) AS fall_2025_cohort_headcount,
    SUM(CASE WHEN e.retained_next_term = TRUE THEN 1 ELSE 0 END) AS retained_to_spring_2026_count,
    ROUND(
        (SUM(CASE WHEN e.retained_next_term = TRUE THEN 1 ELSE 0 END) * 100.0) / NULLIF(COUNT(DISTINCT e.student_id), 0),
        2
    ) AS persistence_rate_pct
FROM banner_enrollment e
WHERE e.term_code = '202510' -- Filters for Fall 2025 cohort
GROUP BY e.program_code
ORDER BY e.program_code;
"""
try:
    result_persistence_rate = pd.read_sql_query(query_persistence_rate, conn)
    print("Persistence Rate (Program-wise) Result:\n", result_persistence_rate)
except Exception as e:
    print(f"Error executing Persistence Rate query: {e}")

Persistence Rate (Program-wise) Result:
         program_code  fall_2025_cohort_headcount  \
0  Arts & Humanities                          17   
1           Business                          20   
2     Data Analytics                          14   
3        Engineering                          18   
4   Natural Sciences                          12   
5            Nursing                          16   

   retained_to_spring_2026_count  persistence_rate_pct  
0                             15                 88.24  
1                             15                 75.00  
2                             11                 78.57  
3                             15                 83.33  
4                             12                100.00  
5                             11                 68.75  


In [91]:
# METRIC: Graduation Rate by Ethnicity and Entry Term Cohort
# Modified to use the new 'graduated' column and 'entry_term' for cohort definition.
# Privacy masking still applies for cohorts < 5.
query_graduation_rate = """
SELECT
    b.ethnicity,
    b.entry_term AS cohort_entry_term,
    CASE
        WHEN COUNT(DISTINCT b.student_id) < 5 THEN NULL
        ELSE COUNT(DISTINCT b.student_id)
    END AS cohort_headcount,
    CASE
        WHEN COUNT(DISTINCT b.student_id) < 5 THEN NULL
        ELSE SUM(CASE WHEN b.graduated = TRUE THEN 1 ELSE 0 END)
    END AS graduates,
    CASE
        WHEN COUNT(DISTINCT b.student_id) < 5 THEN NULL
        ELSE ROUND(
            (SUM(CASE WHEN b.graduated = TRUE THEN 1 ELSE 0 END) * 100.0) / NULLIF(COUNT(DISTINCT b.student_id), 0),
            2
        )
    END AS graduation_rate_pct
FROM banner_enrollment b
WHERE b.term_code = b.entry_term -- Ensure each student is counted only once for their entry term cohort
GROUP BY b.ethnicity, b.entry_term
ORDER BY b.entry_term, b.ethnicity;
"""
try:
    result_graduation_rate = pd.read_sql_query(query_graduation_rate, conn)
    print("Graduation Rate (by Ethnicity and Entry Cohort) Result:\n", result_graduation_rate)
except Exception as e:
    print(f"Error executing Graduation Rate query: {e}")

Graduation Rate (by Ethnicity and Entry Cohort) Result:
                  ethnicity  cohort_entry_term  cohort_headcount  graduates  \
0                    Asian             202510                16          0   
1   Black/African American             202510                23          0   
2          Hispanic/Latino             202510                14          0   
3            International             202510                19          0   
4        Two or More Races             202510                 8          0   
5                    White             202510                17          0   
6                    Asian             202520                22          0   
7   Black/African American             202520                14          0   
8          Hispanic/Latino             202520                19          0   
9            International             202520                14          0   
10       Two or More Races             202520                20          0   
11     

In [93]:
# INTEGRATED VIEW: Cross-System Student Risk & Retention (Enhanced)
# Now includes more detailed student information and more accurate risk-term joining.
query_integrated_view = """
WITH enrollment_status AS (
    SELECT student_id, term_code AS term, credits_attempted AS enrolled_credits, program_code, ethnicity, retained_next_term, graduated, entry_term
    FROM banner_enrollment
),
advising_flags AS (
    SELECT student_id, term_code AS term, risk_level
    FROM navigate_risk_signals
)
SELECT
    e.student_id,
    e.term,
    e.enrolled_credits,
    e.program_code,
    e.ethnicity,
    COALESCE(a.risk_level, 'No Alert') AS risk_level,
    e.retained_next_term,
    e.graduated,
    e.entry_term
FROM enrollment_status e
LEFT JOIN advising_flags a
    ON a.student_id = e.student_id AND a.term = e.term; -- Join on both student and term for accurate risk context
"""
try:
    result_integrated_view = pd.read_sql_query(query_integrated_view, conn)
    print("Integrated View Result (Enhanced):\n", result_integrated_view)
except Exception as e:
    print(f"Error executing Integrated View query: {e}")

Integrated View Result (Enhanced):
     student_id    term  enrolled_credits       program_code  \
0         S001  202510                 9     Data Analytics   
1         S001  202520                14     Data Analytics   
2         S001  202610                16            Nursing   
3         S001  202620                14  Arts & Humanities   
4         S002  202510                16        Engineering   
..         ...     ...               ...                ...   
611       S199  202610                18            Nursing   
612       S199  202620                14            Nursing   
613       S200  202520                 9            Nursing   
614       S200  202610                10            Nursing   
615       S200  202620                10  Arts & Humanities   

                  ethnicity risk_level  retained_next_term  graduated  \
0                     White     Medium                   1          0   
1             International   No Alert                   1   

In [94]:
'''STRATEGIC METRIC: Program Health & Capacity Alignment
   Integrates Enrollment, Risk, Scheduling, and Planning data.
   Includes Small Cell Suppression for FERPA compliance.
'''

program_health= """
WITH enr AS (
    SELECT student_id, term_code, program_code, credits_attempted
    FROM banner_enrollment
),
risk AS (
    SELECT student_id, term_code, risk_level
    FROM navigate_risk_signals
),
sched AS (
    SELECT term_code, course_id, SUM(capacity) AS total_cap, SUM(enrolled_count) AS total_enr
    FROM ad_astra_sections
    GROUP BY term_code, course_id
),
plan AS (
    SELECT term_code, course_id, SUM(planned_sections) AS total_planned
    FROM sectionizer_plans
    GROUP BY term_code, course_id
)

SELECT
    e.term_code,
    e.program_code,

    -- Small Cell Suppression: Mask counts < 5
    CASE
        WHEN COUNT(DISTINCT e.student_id) < 5 THEN NULL
        ELSE COUNT(DISTINCT e.student_id)
    END AS headcount,

    ROUND(AVG(e.credits_attempted), 2) AS avg_credit_load,

    -- At-Risk Momentum Tracking
    CASE
        WHEN COUNT(DISTINCT e.student_id) < 5 THEN NULL
        ELSE COUNT(DISTINCT CASE WHEN r.risk_level IN ('High', 'Medium') THEN e.student_id END)
    END AS at_risk_count,

    -- Capacity Utilization (Fill Rate)
    ROUND((SUM(s.total_enr) * 100.0) / NULLIF(SUM(s.total_cap), 0), 2) AS fill_rate_pct,

    -- Planning Alignment
    SUM(p.total_planned) AS planned_sections

FROM enr e
LEFT JOIN risk r ON e.student_id = r.student_id AND e.term_code = r.term_code
LEFT JOIN sched s ON e.term_code = s.term_code
LEFT JOIN plan p ON e.term_code = p.term_code
GROUP BY e.term_code, e.program_code
ORDER BY e.term_code DESC, headcount DESC;
"""


result_program_health = pd.read_sql_query(program_health, conn)
print("Program Health:\n", result_program_health)


Program Health:
     term_code       program_code  headcount  avg_credit_load  at_risk_count  \
0      202620     Data Analytics         30            11.71              4   
1      202620            Nursing         28            13.36              1   
2      202620   Natural Sciences         23            12.87              4   
3      202620        Engineering         22            14.00              3   
4      202620           Business         22            13.39              3   
5      202620  Arts & Humanities         22            13.96              2   
6      202610            Nursing         32            13.81              6   
7      202610   Natural Sciences         29            13.59              3   
8      202610  Arts & Humanities         28            13.18              6   
9      202610        Engineering         23            14.00              5   
10     202610     Data Analytics         23            12.04              5   
11     202610           Business   

In [95]:
persistence_rate ="""
SELECT
    term_code,
    COUNT(DISTINCT student_id) AS enrolled_students,
    SUM(CASE WHEN retained_next_term = TRUE THEN 1 ELSE 0 END) AS retained_students,
    ROUND(
        (SUM(CASE WHEN retained_next_term = TRUE THEN 1 ELSE 0 END) * 100.0) / NULLIF(COUNT(DISTINCT student_id), 0),
        2
    ) AS persistence_rate_pct
FROM banner_enrollment
GROUP BY term_code
HAVING COUNT(DISTINCT student_id) > 0
ORDER BY term_code;
"""


result_persistence_rate = pd.read_sql_query(persistence_rate, conn)
print("Persistence Rate by Term:\n", result_persistence_rate)

Persistence Rate by Term:
    term_code  enrolled_students  retained_students  persistence_rate_pct
0     202510                 97                 79                 81.44
1     202520                188                147                 78.19
2     202610                154                133                 86.36
3     202620                147                  0                  0.00


In [96]:
'Credit Momentum or Fultime Load'
credit_momentum = """
SELECT
    student_id,
    SUM(credits_attempted) AS total_credits_year
FROM banner_enrollment
WHERE term_code IN ('202510', '202520') -- Filtering for Fall 2025 and Spring 2026
GROUP BY student_id
HAVING SUM(credits_attempted) >= 15;
"""
result_credit_momentum = pd.read_sql_query(credit_momentum, conn)
print("Credit Momentum by year: \n", result_credit_momentum)


Credit Momentum by year: 
     student_id  total_credits_year
0         S001                  23
1         S002                  26
2         S006                  30
3         S008                  18
4         S009                  27
..         ...                 ...
131       S191                  17
132       S192                  29
133       S194                  26
134       S196                  15
135       S198                  16

[136 rows x 2 columns]


In [97]:
' 2 YEAR Completion Tracking'
two_year_completion ="""
WITH StudentEntryAndGraduation AS (
    SELECT
        be.student_id,
        be.entry_term,
        MAX(CASE WHEN be.graduated = TRUE THEN be.term_code ELSE NULL END) AS actual_graduated_term
    FROM banner_enrollment be
    GROUP BY be.student_id, be.entry_term
)
SELECT
    substr(scg.entry_term, 1, 4) AS cohort_year,
    scg.entry_term AS cohort_start_term,
    COUNT(DISTINCT scg.student_id) AS cohort_size,
    COUNT(DISTINCT CASE
        WHEN scg.actual_graduated_term IS NOT NULL
             -- Assuming 'YYYYMM' term format, (grad_term - entry_term) <= 200 roughly means within 2 years.
             -- E.g., 202710 (Fall 2027) - 202510 (Fall 2025) = 200
             AND (CAST(scg.actual_graduated_term AS INTEGER) - CAST(scg.entry_term AS INTEGER)) <= 200
        THEN scg.student_id
    END) AS completed_in_2_years
FROM StudentEntryAndGraduation scg
GROUP BY cohort_year, scg.entry_term
ORDER BY scg.entry_term;
"""

result_completion= pd.read_sql_query(two_year_completion, conn)
print("Two Year Completion Tracking for AAS: \n ", result_completion)


Two Year Completion Tracking for AAS: 
    cohort_year  cohort_start_term  cohort_size  completed_in_2_years
0        2025             202510           97                     0
1        2025             202520          103                     0


In [98]:
'Natural Science Transfer Student'
transfer_student_capture = """
-- Assuming 'Natural Sciences' is the target AAS program and 'is_transfer' identifies transfer students.
SELECT
    CAST(COUNT(DISTINCT CASE WHEN b.is_transfer = TRUE THEN b.student_id END) AS REAL) AS total_transfer_students,
    COUNT(DISTINCT CASE WHEN b.is_transfer = TRUE AND b.program_code = 'Natural Sciences' AND b.graduated = TRUE THEN b.student_id END) AS transfer_natural_science_graduates,
    ROUND(
        (COUNT(DISTINCT CASE WHEN b.is_transfer = TRUE AND b.program_code = 'Natural Sciences' AND b.graduated = TRUE THEN b.student_id END) * 100.0) / NULLIF(COUNT(DISTINCT CASE WHEN b.is_transfer = TRUE THEN b.student_id END), 0),
        2
    ) AS natural_science_aas_transfer_rate_pct
FROM banner_enrollment b;
"""
results_transfer_student = pd.read_sql_query (transfer_student_capture, conn)
print ('Natural Science students tranfered: \n', results_transfer_student)


Natural Science students tranfered: 
    total_transfer_students  transfer_natural_science_graduates  \
0                     43.0                                   0   

   natural_science_aas_transfer_rate_pct  
0                                    0.0  


In [99]:
' Certificate Capture Rate'
certificate_capture = """
SELECT
    term_code,
    certificate_program,
    COUNT(DISTINCT student_id) AS students_in_pathway,
    SUM(CASE WHEN certificate_status = 'Completed' THEN 1 ELSE 0 END) AS students_completed_cert,
    ROUND(
        (SUM(CASE WHEN certificate_status = 'Completed' THEN 1 ELSE 0 END) * 100.0) / NULLIF(COUNT(DISTINCT student_id), 0),
        2
    ) AS certificate_capture_rate_pct
FROM banner_enrollment
WHERE certificate_program IS NOT NULL
GROUP BY term_code, certificate_program
ORDER BY term_code, certificate_program;
"""

result_cert_capture= pd.read_sql_query(certificate_capture, conn)
print("Certificate Capture Rate:\n", result_cert_capture)


Certificate Capture Rate:
     term_code      certificate_program  students_in_pathway  \
0      202510       Cybersecurity Cert                    9   
1      202510  Data Science Micro-Cert                    6   
2      202510    Healthcare Admin Cert                    9   
3      202510     Web Development Cert                   11   
4      202520       Cybersecurity Cert                   15   
5      202520  Data Science Micro-Cert                   15   
6      202520    Healthcare Admin Cert                   20   
7      202520     Web Development Cert                   21   
8      202610       Cybersecurity Cert                   14   
9      202610  Data Science Micro-Cert                   13   
10     202610    Healthcare Admin Cert                   13   
11     202610     Web Development Cert                   18   
12     202620       Cybersecurity Cert                   14   
13     202620  Data Science Micro-Cert                   13   
14     202620    Healthcare 